# mixed4e:55 reproduction

Independent reproduction of neuron-level clustering on InceptionV1 `mixed4e:55`. Method: [docs/thesis/method.md](../docs/thesis/method.md), implementation notes in [docs/implementation/mixed4e_55.md](../docs/implementation/mixed4e_55.md). All logic lives in the `neural_interp` package; this notebook only runs it on a Colab GPU and stores results in `gs://pj-okc/neural_interp/mixed4e_55/`.

In [ ]:
REF = "main"
!pip install -q "neural-interp @ git+https://github.com/priyanshujain/neural-interp@{REF}"

In [ ]:
from google.colab import auth
auth.authenticate_user()
!mkdir -p /content/data /content/reference
!test -d /content/data/inception_pool || (gcloud storage cp gs://pj-okc/inception_pool.tar /content/ && tar --warning=no-unknown-keyword -xf /content/inception_pool.tar -C /content/data && find /content/data -name '._*' -delete)
!test -d /content/reference/runs || (gcloud storage cp gs://pj-okc/mixed4e_55/mixed4e_55_full.tar /content/ && tar -xf /content/mixed4e_55_full.tar -C /content/reference runs/mixed4e_55/space)

## Same images as the reference run

The reference run's 2,000 image ids are read from its saved hits, so clusters can be compared hit by hit.

In [ ]:
import numpy as np
hits = np.load('/content/reference/runs/mixed4e_55/space/hits.npz', allow_pickle=True)
open('/content/sample_ids.txt', 'w').write('\n'.join(hits['sample_ids']))
!nint collect --pool /content/data/inception_pool --image-list /content/sample_ids.txt --out /content/runs/same_images

In [ ]:
!nint report --pool /content/data/inception_pool --out /content/runs/same_images
!nint compare --ours /content/runs/same_images/space --reference /content/reference/runs/mixed4e_55/space --out /content/runs/same_images/compare.json

## Fresh sample

Our own random 2,000 images from the same pool, to check the result does not depend on the sample.

In [ ]:
!nint collect --pool /content/data/inception_pool --seed 1 --out /content/runs/fresh_sample
!nint report --pool /content/data/inception_pool --seed 1 --out /content/runs/fresh_sample
!nint compare --ours /content/runs/fresh_sample/space --reference /content/reference/runs/mixed4e_55/space --out /content/runs/fresh_sample/compare.json

## Results

In [ ]:
import json
for run in ['same_images', 'fresh_sample']:
    c = json.load(open(f'/content/runs/{run}/compare.json'))
    print(run, 'shared hits', c['shared_hits'], 'ARI', c['ari_on_shared'])
    for side in ['ours', 'reference']:
        w = c[side]
        print(f"  {side}: {w['hits']} hits, {w['clusters']} clusters, weighted purity {w['weighted_purity']:.2f}, groups {({k: round(v, 2) for k, v in w['group_share'].items()})}")

In [ ]:
!gcloud storage cp -r /content/runs gs://pj-okc/neural_interp/mixed4e_55/